# Dataset from
https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed/data

In [ ]:
!pip install -q mediapipe==0.10.7

ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.7 (from versions: 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)

[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for mediapipe==0.10.7


In [1]:
import pandas as pd
import plotly.express as px
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
%matplotlib inline

import os, json, gc, pickle
import cv2
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import mediapipe as mp


sns.set_style('darkgrid')
matplotlib.rcParams["font.size"]=14
matplotlib.rcParams["figure.figsize"] = (10,6)
pd.set_option('display.max_columns', None)
matplotlib.rcParams["figure.facecolor"] = "#00000000"

In [2]:
DATA_DIR = os.path.join("datasets", "WLASL")
MODEL_PATH = "hand_landmarker.task"
VIDEO_DIR = os.path.join(DATA_DIR, "videos")
NSLT_FILE = os.path.join(DATA_DIR, "nslt_100.json")
CLASS_FILE = os.path.join(DATA_DIR, "wlasl_class_list.txt")
FEATURES_FILE = os.path.join(DATA_DIR, "features_100.npz")

NUM_FRAMES = 25          # evenly-spaced frames per video
HAND_LANDMARKS = 21       # 21 landmsrks per hand
MAX_HANDS = 2             # extract both hands
FEATURE_DIM = MAX_HANDS * HAND_LANDMARKS * 3  # 126 per frame
# after aggregate (mean+std): 126*2 = 252

In [3]:
# ### Cell 2: Load labels and class names
# Load class list: "0\tbook" -> {0: "book", ...}
class_map = {}
with open(CLASS_FILE, "r") as f:
    for line in f:
        idx, word = line.strip().split("\t")
        class_map[int(idx)] = word

print(f"Loaded {len(class_map)} classes from wlasl_class_list.txt")

# Load nslt_100.json -> {video_id: {subset, action: [class_id, ?, ?]}}
with open(NSLT_FILE, "r") as f:
    nslt_data = json.load(f)

print(f"Loaded {len(nslt_data)} samples from nslt_100.json")

# Build video_id -> (subset, word) mapping
video_labels = {}
for video_id, info in nslt_data.items():
    class_id = info["action"][0]
    word = class_map.get(class_id, f"unknown_{class_id}")
    video_labels[video_id] = {
        "subset": info["subset"],
        "class_id": class_id,
        "word": word
    }


Loaded 2000 classes from wlasl_class_list.txt
Loaded 2038 samples from nslt_100.json


In [4]:
# Count by split
from collections import Counter
splits = Counter(v["subset"] for v in video_labels.values())
print(f"Split counts: train={splits.get('train',0)}, val={splits.get('val',0)}, test={splits.get('test',0)}")
print(f"Unique classes: {len(set(v['word'] for v in video_labels.values()))}")

# Check video availability
videos_on_disk = set(f.replace(".mp4", "") for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4"))
available = [vid for vid in video_labels if vid in videos_on_disk]
print(f"Videos available on disk: {len(available)} / {len(video_labels)}")


Split counts: train=1442, val=338, test=258
Unique classes: 100
Videos available on disk: 1013 / 2038


In [5]:
# ### Cell 3: Initialize MediaPipe
from mediapipe.tasks.python import vision
options = vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
)

# %%
base_opts = mp.tasks.BaseOptions(model_asset_path=MODEL_PATH)
hand_options = vision.HandLandmarkerOptions(
    base_options=base_opts,
    running_mode=vision.RunningMode.IMAGE,   # individual frames, not streaming
    num_hands=MAX_HANDS,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.1,
    min_tracking_confidence=0.1
)
landmarker = vision.HandLandmarker.create_from_options(hand_options)

In [6]:
# ### Cell 4: Feature extraction function (Tasks API)

# %%
def extract_features(video_id, num_frames=NUM_FRAMES):
    """
    Extract hand landmarks from evenly-spaced frames using MediaPipe Tasks API.
    Returns feature vector of shape (252,) and the word label.
    """
    video_path = os.path.join(VIDEO_DIR, f"{video_id}.mp4")
    if not os.path.exists(video_path):
        return None, None

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < 1:
        cap.release()
        return None, None

    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    all_landmarks = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)

    for i, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        result = landmarker.detect(mp_image)       # IMAGE mode

        if result.hand_landmarks:
            for h_idx, hand_lm in enumerate(result.hand_landmarks[:MAX_HANDS]):
                for lm_idx, lm in enumerate(hand_lm):
                    offset = h_idx * HAND_LANDMARKS * 3
                    all_landmarks[i, offset + lm_idx * 3 + 0] = lm.x
                    all_landmarks[i, offset + lm_idx * 3 + 1] = lm.y
                    all_landmarks[i, offset + lm_idx * 3 + 2] = lm.z

    cap.release()

    return all_landmarks, video_labels[video_id]["word"]  # shape (15, 126)


# Quick test
test_vid = available[0]
feat, label = extract_features(test_vid)
print(f"Test: {test_vid}  →  {label}")
print(f"Feature shape: {feat.shape}  Non-zero: {(feat != 0).sum()}/{len(feat)}")

Test: 69422  →  orange
Feature shape: (25, 126)  Non-zero: 1071/25


In [7]:
# ### Cell 5: Extract features for all videos (or load cached)

# %%
if os.path.exists(FEATURES_FILE):
    print("Loading cached features...")
    data = np.load(FEATURES_FILE, allow_pickle=True)
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_val = data["X_val"]
    y_val = data["y_val"]
    X_test = data["X_test"]
    y_test = data["y_test"]
    print(f"Loaded: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
else:
    print("Extracting features (~20-30 min)...")
    X_train_list, y_train_list = [], []
    X_val_list, y_val_list = [], []
    X_test_list, y_test_list = [], []
    skipped = 0

    for vid in tqdm(available, desc="Extracting", ncols=80):
        feat, word = extract_features(vid)
        if feat is None:
            skipped += 1
            continue
        subset = video_labels[vid]["subset"]
        if subset == "train":
            X_train_list.append(feat)
            y_train_list.append(word)
        elif subset == "val":
            X_val_list.append(feat)
            y_val_list.append(word)
        else:
            X_test_list.append(feat)
            y_test_list.append(word)

    X_train = np.array(X_train_list, dtype=np.float32)
    y_train = np.array(y_train_list)
    X_val = np.array(X_val_list, dtype=np.float32)
    y_val = np.array(y_val_list)
    X_test = np.array(X_test_list, dtype=np.float32)
    y_test = np.array(y_test_list)
    print(f"\nExtracted: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, skipped={skipped}")
    np.savez_compressed(FEATURES_FILE,
                        X_train=X_train, y_train=y_train,
                        X_val=X_val, y_val=y_val,
                        X_test=X_test, y_test=y_test)
    print(f"Saved to {FEATURES_FILE}")



Extracting features (~20-30 min)...


Extracting: 100%|███████████████████████████| 1013/1013 [28:46<00:00,  1.70s/it]



Extracted: train=748, val=165, test=100, skipped=0
Saved to datasets\WLASL\features_100.npz


In [8]:
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

model = tf.keras.Sequential(
    [
        tf.keras.layers.Masking(mask_value=0.0, input_shape=(15, 126)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(len(le.classes_), activation="softmax"),
    ]
)

model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
model.fit(
    X_train,
    y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=100,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    ],
)

Epoch 1/100


C:\Users\Fawad Arshad\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.0174 - loss: 4.6081 - val_accuracy: 0.0121 - val_loss: 4.5485
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.0241 - loss: 4.4876 - val_accuracy: 0.0182 - val_loss: 4.4274
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0348 - loss: 4.3674 - val_accuracy: 0.0303 - val_loss: 4.3111
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0281 - loss: 4.2423 - val_accuracy: 0.0303 - val_loss: 4.2344
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.0334 - loss: 4.1444 - val_accuracy: 0.0485 - val_loss: 4.1510
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.0388 - loss: 4.0942 - val_accuracy: 0.0424 - val_loss: 4.0955
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.0374 - loss: 4.0249 - val_accuracy: 0.0545 - val_loss: 4.0553
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.0441 - loss: 3.9399 - val_accuracy: 0.0424 - val_l